# Chapter 51: Scaling AI Systems

**Volume 3 — Advanced Techniques for Production**

10 users: works fine. 50 users: getting slow. 500 users: paged at 2am.

**The naive fix:** "add more servers."
**The real question:** "what is the actual bottleneck?"

LLM workloads have a completely different bottleneck profile than web services.
Misdiagnose it and you waste money on hardware that doesn't help.

### What You Will Build
1. **Worker Pool + Backpressure** — bounded queue, dead letter queue, at-least-once delivery
2. **Load Balancer Strategies** — round-robin vs least-connections vs random, live comparison
3. **Priority Queue** — real-time NOC queries win over batch jobs; queue depth as scaling signal
4. **Request Batching** — static vs dynamic vs continuous batching throughput comparison
5. **Reactive Autoscaler** — scale out on queue depth + Little's Law capacity planning

### The Core Insight
```
Web service bottleneck:  CPU (add cores -> linear speedup)
LLM prefill bottleneck:  GPU FLOP/s (compute-bound)
LLM decode bottleneck:   GPU memory bandwidth (memory-bound)
-> Adding GPU compute during decode does NOT help!
```

In [ ]:
# pip install anthropic
# All scaling primitives built from Python standard library!

import os, json, time, math, random, threading
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import Optional
import queue as queue_module   # stdlib queue (not to clash with variable names)

# ── Anthropic client ───────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("Anthropic API connected")
else:
    _client = None
    print("No API key - simulation mode (all scaling patterns still work!)")

MODEL = "claude-haiku-4-5-20251001"

# ── Simulated LLM call: realistic latency + tokens ────────────────────────────
def llm_call(prompt: str, max_tokens: int = 150) -> tuple:
    'Returns (response_text, latency_ms, tokens_in, tokens_out).'
    start = time.time()
    if USE_API:
        resp = client.messages.create(
            model=MODEL, max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        text = resp.content[0].text
        tok_in  = resp.usage.input_tokens
        tok_out = resp.usage.output_tokens
    else:
        # Simulate realistic LLM latency (50-300ms scaled for demo)
        time.sleep(random.uniform(0.03, 0.15))
        p = prompt.lower()
        if "bgp" in p:
            text = "BGP path selection: LOCAL_PREF > AS_PATH > MED > eBGP/iBGP > IGP metric > Router-ID."
        elif "ospf" in p:
            text = "OSPF DR election: highest priority wins, then highest Router-ID. Elected on broadcast/NBMA only."
        elif "qos" in p:
            text = "QoS DSCP: EF (46) for voice, AF41 for video, CS3 for call signaling, BE for best-effort."
        else:
            text = f"Network answer for: {prompt[:50]}..."
        tok_in  = len(prompt.split()) + 15
        tok_out = len(text.split())

    latency = (time.time() - start) * 1000
    return text, latency, tok_in, tok_out


# ── Job dataclass used across all demos ───────────────────────────────────────
@dataclass
class Job:
    job_id: int
    prompt: str
    priority: int = 1           # 1=low (batch), 2=medium, 3=high (real-time)
    retries: int = 0
    max_retries: int = 3
    result: Optional[str] = None
    error: Optional[str] = None
    latency_ms: float = 0.0
    worker_id: Optional[int] = None
    enqueue_time: float = field(default_factory=time.time)
    complete_time: Optional[float] = None

    def wait_time_ms(self) -> float:
        if self.complete_time:
            return (self.complete_time - self.enqueue_time) * 1000
        return 0.0

print(f"Model: {MODEL}")
print("Ready. Building 5 production scaling patterns from scratch!")

## Demo 1: Worker Pool + Queue + Backpressure + Dead Letter Queue

**The pattern:** separate the API receiver from the LLM workers with a queue.
- API receives request -> puts Job into queue -> returns `job_id` immediately
- Workers pull from queue -> call LLM -> store result

**Backpressure:** queue has a max size. When full, new requests get `429 Too Many Requests`
instead of waiting forever. Clients must back off and retry — this is healthy!

**Dead Letter Queue (DLQ):** if a job fails 3 times, move it to DLQ instead of
retrying forever. Engineers inspect the DLQ to find root causes.

> **In Simple Words:**
> Worker pool queue = the ticket queue in your NOC.
> Backpressure = "sorry, max 50 tickets open at once — fix existing ones first."
> DLQ = "this ticket keeps crashing the system, move it to HOLD for investigation." 

In [ ]:
# ── Demo 1: Worker Pool + Backpressure + Dead Letter Queue ────────────────────

class WorkerPool:
    '''
    Thread-based AI worker pool with:
    - Bounded queue (backpressure)
    - Multiple concurrent workers
    - Dead letter queue after max_retries failures
    - Per-worker stats
    '''
    def __init__(self, num_workers: int = 3, queue_max: int = 10):
        self.num_workers  = num_workers
        self.queue_max    = queue_max
        self._queue       = queue_module.Queue(maxsize=queue_max)
        self._dlq         = []           # dead letter queue (failed jobs)
        self._completed   = []
        self._stats       = defaultdict(lambda: {"processed": 0, "errors": 0,
                                                  "total_ms": 0.0})
        self._lock        = threading.Lock()
        self._workers     = []
        self._running     = False
        self._job_counter = 0

    def start(self):
        self._running = True
        for i in range(self.num_workers):
            t = threading.Thread(target=self._worker_loop, args=(i,), daemon=True)
            t.start()
            self._workers.append(t)

    def stop(self):
        self._running = False
        # Poison pills - one per worker to unblock them
        for _ in self._workers:
            try:
                self._queue.put(None, timeout=1)
            except queue_module.Full:
                pass

    def submit(self, prompt: str, priority: int = 1) -> tuple:
        'Submit a job. Returns (job_id, "queued") or (-1, "backpressure_429").'
        if self._queue.full():
            return -1, "backpressure_429"
        with self._lock:
            self._job_counter += 1
            job_id = self._job_counter
        job = Job(job_id=job_id, prompt=prompt, priority=priority)
        self._queue.put(job)
        return job_id, "queued"

    def _worker_loop(self, worker_id: int):
        while self._running:
            try:
                job = self._queue.get(timeout=0.5)
            except queue_module.Empty:
                continue
            if job is None:   # poison pill
                break

            job.worker_id = worker_id
            try:
                text, lat_ms, _, _ = llm_call(job.prompt)
                job.result = text
                job.latency_ms = lat_ms
                job.complete_time = time.time()
                with self._lock:
                    self._completed.append(job)
                    s = self._stats[worker_id]
                    s["processed"] += 1
                    s["total_ms"] += lat_ms

            except Exception as e:
                job.retries += 1
                job.error = str(e)[:60]
                if job.retries >= job.max_retries:
                    job.complete_time = time.time()
                    with self._lock:
                        self._dlq.append(job)
                        self._stats[worker_id]["errors"] += 1
                else:
                    # Requeue with exponential backoff simulation
                    time.sleep(0.05 * (2 ** job.retries))
                    self._queue.put(job)

            finally:
                self._queue.task_done()

    def queue_depth(self) -> int:
        return self._queue.qsize()

    def report(self) -> dict:
        with self._lock:
            total = len(self._completed)
            dlq   = len(self._dlq)
            if total:
                avg_lat = sum(j.latency_ms for j in self._completed) / total
                avg_wait = sum(j.wait_time_ms() for j in self._completed) / total
            else:
                avg_lat = avg_wait = 0
        return {
            "completed":       total,
            "dlq_failures":    dlq,
            "avg_latency_ms":  round(avg_lat, 1),
            "avg_wait_ms":     round(avg_wait, 1),
            "worker_stats":    dict(self._stats),
        }


# ── Run the demo ───────────────────────────────────────────────────────────────
print("=== Worker Pool Demo (3 workers, queue max=10) ===")
pool = WorkerPool(num_workers=3, queue_max=10)
pool.start()

# Submit 15 jobs - first 10 will queue, rest will get backpressure
bgp_queries = [
    "What is LOCAL_PREF in BGP?",
    "Explain BGP AS_PATH attribute",
    "How does BGP MED work?",
    "What is BGP NEXT_HOP?",
    "Explain OSPF DR election",
    "What is OSPF area 0?",
    "Describe QoS DSCP EF",
    "What is BGP ORIGIN attribute?",
    "Explain OSPF hello packets",
    "What does BGP COMMUNITY do?",
    "BGP weight attribute explained",
    "OSPF LSA types overview",
    "QoS traffic shaping vs policing",
    "BGP route reflector purpose",
    "OSPF stub area benefits",
]

print()
print(f"Submitting {len(bgp_queries)} jobs to queue (max size=10):")
queued = backpressured = 0
for i, q in enumerate(bgp_queries):
    job_id, status = pool.submit(q)
    if status == "queued":
        queued += 1
    else:
        backpressured += 1
    marker = "OK" if status == "queued" else "429 BACKPRESSURE"
    print(f"  Job {i+1:2d}: [{marker}]  queue_depth={pool.queue_depth()}")

print()
print(f"Queued: {queued}, Backpressure (429): {backpressured}")
print("Workers processing... (wait 2 seconds)")
time.sleep(2.0)
pool.stop()
time.sleep(0.3)

report = pool.report()
print()
print("=== Results ===")
print(f"  Completed:         {report['completed']}")
print(f"  DLQ failures:      {report['dlq_failures']}")
print(f"  Avg LLM latency:   {report['avg_latency_ms']:.0f} ms")
print(f"  Avg queue wait:    {report['avg_wait_ms']:.0f} ms")
print()
print("Per-worker stats:")
for wid, stats in sorted(report['worker_stats'].items()):
    avg = stats['total_ms'] / max(stats['processed'], 1)
    print(f"  Worker {wid}: {stats['processed']:2d} jobs processed, "
          f"{stats['errors']} errors, avg {avg:.0f}ms/job")

# ── Little's Law capacity planning ────────────────────────────────────────────
print()
print("=== Little's Law Capacity Planning ===")
print("L = lambda * W   (L=queue length, lambda=arrival rate, W=service time)")
print()
service_time_s = report['avg_latency_ms'] / 1000 if report['avg_latency_ms'] else 0.1
scenarios = [
    ("Low traffic",    2,  3),
    ("Normal traffic", 5,  3),
    ("High traffic",   8,  3),
    ("Peak traffic",  12,  3),
    ("Overloaded",    20,  3),
]
print(f"{'Scenario':<18} {'Arrivals/s':>11} {'Workers':>8} {'Queue depth L':>14} {'Wait (s)':>9} {'Status'}")
print("-" * 70)
for name, arrivals_s, workers in scenarios:
    throughput = workers / service_time_s   # max jobs/s with N workers
    if arrivals_s <= throughput:
        L = arrivals_s * service_time_s
        wait = L / arrivals_s
        status = "OK"
    else:
        L = float('inf')
        wait = float('inf')
        status = "SATURATED - scale out!"
    L_str    = f"{L:.1f}" if L != float('inf') else "growing..."
    wait_str = f"{wait:.2f}s" if wait != float('inf') else "infinite"
    print(f"{name:<18} {arrivals_s:>11} {workers:>8} {L_str:>14} {wait_str:>9}  {status}")

## Demo 2: Load Balancer Strategies — Round-Robin vs Least-Connections vs Random

A load balancer distributes incoming requests across multiple worker instances.
The strategy determines which instance gets each request.

| Strategy | How It Works | Best For |
|----------|-------------|---------|
| **Round-Robin** | Worker 0, 1, 2, 0, 1, 2... | Uniform request latency |
| **Least-Connections** | Send to worker with fewest active tasks | Mixed short/long requests |
| **Random** | Pick any worker at random | Simple, stateless, surprisingly effective |

> **In Simple Words:** Load balancer strategies = routing protocols.
> Round-robin = static routing (equal cost, take turns).
> Least-connections = dynamic routing based on real-time utilization.
> Random = anycast — pick any closest node.

In [ ]:
# ── Demo 2: Load Balancer Strategies ─────────────────────────────────────────

@dataclass
class WorkerInstance:
    'Represents one AI service instance behind the load balancer.'
    instance_id: int
    active_connections: int = 0
    total_handled: int = 0
    total_latency_ms: float = 0.0

    def avg_latency(self) -> float:
        return self.total_latency_ms / self.total_handled if self.total_handled else 0.0


class LoadBalancer:
    '''
    Simulates a load balancer with pluggable routing strategies.
    In production: nginx (round-robin/least-conn) or HAProxy (many options).
    '''
    def __init__(self, num_instances: int, strategy: str = "round_robin"):
        self.strategy    = strategy
        self.instances   = [WorkerInstance(i) for i in range(num_instances)]
        self._rr_index   = 0
        self._lock       = threading.Lock()
        self.rejected    = 0    # requests rejected due to all workers busy

    def _pick_round_robin(self) -> WorkerInstance:
        with self._lock:
            w = self.instances[self._rr_index % len(self.instances)]
            self._rr_index += 1
        return w

    def _pick_least_connections(self) -> WorkerInstance:
        with self._lock:
            return min(self.instances, key=lambda w: w.active_connections)

    def _pick_random(self) -> WorkerInstance:
        return random.choice(self.instances)

    def pick(self) -> WorkerInstance:
        if self.strategy == "round_robin":
            return self._pick_round_robin()
        elif self.strategy == "least_connections":
            return self._pick_least_connections()
        else:
            return self._pick_random()

    def handle_request(self, latency_ms: float):
        'Simulate a request: pick worker, "process" for latency_ms, release.'
        w = self.pick()
        with self._lock:
            w.active_connections += 1
        time.sleep(latency_ms / 1000)   # scaled down for demo
        with self._lock:
            w.active_connections -= 1
            w.total_handled += 1
            w.total_latency_ms += latency_ms

    def report(self) -> dict:
        total = sum(w.total_handled for w in self.instances)
        dist  = [w.total_handled for w in self.instances]
        # Imbalance = max deviation from ideal even distribution
        ideal = total / len(self.instances) if self.instances else 1
        max_dev = max(abs(w.total_handled - ideal) for w in self.instances)
        imbalance_pct = (max_dev / ideal * 100) if ideal else 0
        return {
            "strategy":         self.strategy,
            "total_requests":   total,
            "per_instance":     dist,
            "imbalance_pct":    round(imbalance_pct, 1),
            "avg_latencies":    [round(w.avg_latency(), 1) for w in self.instances],
        }


def run_lb_simulation(strategy: str, request_latencies: list,
                      num_instances: int = 4) -> dict:
    'Run concurrent requests through a load balancer and measure distribution.'
    lb = LoadBalancer(num_instances, strategy=strategy)
    threads = []
    for lat in request_latencies:
        t = threading.Thread(target=lb.handle_request, args=(lat,))
        threads.append(t)
    start = time.time()
    for t in threads: t.start()
    for t in threads: t.join()
    elapsed = (time.time() - start) * 1000
    r = lb.report()
    r["wall_time_ms"] = round(elapsed, 0)
    return r


# ── Simulate mixed traffic: some fast requests, some slow ─────────────────────
random.seed(42)
# Mix of fast (50-200ms) and slow (500-2000ms) requests - realistic LLM traffic
request_latencies = (
    [random.uniform(5, 20) for _ in range(60)] +     # fast requests (scaled down)
    [random.uniform(50, 200) for _ in range(20)]      # slow requests (scaled down)
)
random.shuffle(request_latencies)

print("=== Load Balancer Strategy Comparison ===")
print(f"Sending {len(request_latencies)} requests (mix of fast+slow) to 4 worker instances")
print()

strategies = ["round_robin", "least_connections", "random"]
results = {}
for strategy in strategies:
    r = run_lb_simulation(strategy, request_latencies, num_instances=4)
    results[strategy] = r

print(f"{'Strategy':<22} {'Wall time':>10} {'Per instance':>35} {'Imbalance%':>12}")
print("-" * 82)
for strategy, r in results.items():
    per_inst = str(r["per_instance"])
    print(f"{strategy:<22} {r['wall_time_ms']:>9.0f}ms {per_inst:>35} {r['imbalance_pct']:>11.1f}%")

print()
print("Key observations:")
print("  Round-robin:        Equal count distribution, but ignores actual load")
print("  Least-connections:  Better for mixed fast/slow requests, fewer idle workers")
print("  Random:             Surprisingly competitive for large N (law of large numbers)")

# ── Amdahl's Law demonstration ────────────────────────────────────────────────
print()
print("=== Amdahl's Law: Theoretical Scaling Limits ===")
print("LLM decode is sequential (each token depends on previous) -> can't parallelize")
print()

def amdahls_speedup(n_workers: int, sequential_fraction: float) -> float:
    'Maximum speedup with n_workers when sequential_fraction cannot be parallelized.'
    return 1.0 / (sequential_fraction + (1 - sequential_fraction) / n_workers)

print(f"{'Workers':>8} {'s=0% seq':>10} {'s=10% seq':>12} {'s=20% seq':>12} {'LLM decode (s=60%)':>20}")
print("-" * 68)
for n in [1, 2, 4, 8, 16, 32, 64]:
    s0  = amdahls_speedup(n, 0.0)
    s10 = amdahls_speedup(n, 0.10)
    s20 = amdahls_speedup(n, 0.20)
    s60 = amdahls_speedup(n, 0.60)   # LLM decode heavily sequential
    print(f"{n:>8}x   {s0:>9.1f}x   {s10:>11.1f}x   {s20:>11.1f}x   {s60:>19.1f}x")
print()
print("LLM decode at s=60%: max theoretical speedup = 2.5x (no matter how many GPUs!)")
print("Solution: batching (process many requests simultaneously) not single-request speed")

## Demo 3: Priority Queue — Real-Time Wins Over Batch

Not all AI requests are equal:
- **Priority 3 (HIGH):** NOC engineer asks live question during incident — needs answer NOW
- **Priority 2 (MED):** Automated monitoring check — important but not instant
- **Priority 1 (LOW):** Overnight batch audit of 5,000 devices — can wait

A priority queue ensures real-time queries **jump the line** over batch jobs.
Workers always drain the highest-priority items first.

> **In Simple Words:** Priority queue = QoS queuing on a router.
> Voice traffic (EF) = real-time queries — always served first.
> Best-effort = batch jobs — served when there's capacity.
> PQ scheduling = "never touch batch until real-time queue is empty."

**Queue depth as scaling signal:**
- Queue depth growing = workers can't keep up -> scale out NOW
- Queue depth draining = capacity sufficient -> hold or scale in

In [ ]:
# ── Demo 3: Priority Queue with Queue Depth Monitoring ───────────────────────

import heapq

class PriorityJobQueue:
    '''
    Min-heap priority queue (lower number = higher urgency for us, so we negate priority).
    Workers always pick the highest-priority pending job.

    Priority levels:
        3 = HIGH  (real-time NOC queries)
        2 = MED   (automated monitoring)
        1 = LOW   (batch config audit)
    '''
    def __init__(self, max_size: int = 100):
        self._heap   = []       # (neg_priority, enqueue_time, job)
        self._lock   = threading.Lock()
        self._not_empty = threading.Event()
        self.max_size = max_size
        self._rejected = 0
        self._counter  = 0    # tie-breaker for equal priority

    def put(self, job: Job) -> bool:
        'Returns True if enqueued, False if full (backpressure).'
        with self._lock:
            if len(self._heap) >= self.max_size:
                self._rejected += 1
                return False
            self._counter += 1
            heapq.heappush(self._heap, (-job.priority, self._counter, job))
            self._not_empty.set()
            return True

    def get(self, timeout: float = 0.5) -> Optional[Job]:
        self._not_empty.wait(timeout=timeout)
        with self._lock:
            if not self._heap:
                self._not_empty.clear()
                return None
            _, _, job = heapq.heappop(self._heap)
            if not self._heap:
                self._not_empty.clear()
            return job

    def depth(self) -> int:
        return len(self._heap)

    def depth_by_priority(self) -> dict:
        with self._lock:
            counts = defaultdict(int)
            for neg_p, _, _ in self._heap:
                counts[-neg_p] += 1
        return dict(counts)


class PriorityWorkerPool:
    def __init__(self, num_workers: int = 2, queue_max: int = 30):
        self._pq         = PriorityJobQueue(max_size=queue_max)
        self._completed  = []
        self._lock       = threading.Lock()
        self._running    = False
        self.num_workers = num_workers
        self._depth_log  = []   # (timestamp, depth) for scaling signal

    def start(self):
        self._running = True
        for i in range(self.num_workers):
            threading.Thread(target=self._worker, args=(i,), daemon=True).start()
        threading.Thread(target=self._depth_monitor, daemon=True).start()

    def stop(self):
        self._running = False

    def submit(self, prompt: str, priority: int = 1) -> bool:
        with self._lock:
            job_id = len(self._completed) + self._pq.depth() + 1
        job = Job(job_id=job_id, prompt=prompt, priority=priority)
        return self._pq.put(job)

    def _worker(self, worker_id: int):
        priority_labels = {3: "HIGH", 2: "MED", 1: "LOW"}
        while self._running:
            job = self._pq.get(timeout=0.3)
            if job is None:
                continue
            _, lat_ms, _, _ = llm_call(job.prompt)
            job.latency_ms = lat_ms
            job.complete_time = time.time()
            job.worker_id = worker_id
            with self._lock:
                self._completed.append(job)
                label = priority_labels.get(job.priority, "?")
                print(f"  Worker-{worker_id} [{label:>4}] job-{job.job_id:2d} "
                      f"done in {lat_ms:.0f}ms | q_depth={self._pq.depth()}")

    def _depth_monitor(self):
        while self._running:
            time.sleep(0.1)
            depth = self._pq.depth()
            self._depth_log.append((time.time(), depth))

    def stats(self) -> dict:
        with self._lock:
            if not self._completed:
                return {}
            by_priority = defaultdict(list)
            for j in self._completed:
                by_priority[j.priority].append(j.latency_ms)
            pstats = {}
            labels = {3: "HIGH (real-time)", 2: "MED (monitoring)", 1: "LOW (batch)"}
            for p in sorted(by_priority.keys(), reverse=True):
                lats = by_priority[p]
                pstats[labels[p]] = {
                    "count": len(lats),
                    "avg_latency_ms": round(sum(lats)/len(lats), 0)
                }
            return {"completed": len(self._completed), "by_priority": pstats,
                    "queue_rejected": self._pq._rejected}


# ── Submit mixed priority jobs ────────────────────────────────────────────────
pool3 = PriorityWorkerPool(num_workers=2, queue_max=30)
pool3.start()

print("=== Priority Queue Demo (2 workers) ===")
print("Submitting jobs in this order: 5 batch (LOW), then 3 real-time (HIGH), then 2 monitoring (MED)")
print()

# First: flood with LOW priority batch jobs
for i in range(5):
    pool3.submit(f"Batch audit item {i+1}: check OSPF area config on device-{i+1}", priority=1)

# Then: real-time NOC queries (should jump the queue)
time.sleep(0.05)
for i in range(3):
    pool3.submit(f"URGENT: BGP neighbor down on core-router-{i+1}?", priority=3)

# Then: medium priority monitoring
time.sleep(0.05)
for i in range(2):
    pool3.submit(f"Monitoring check: QoS policy status on dist-switch-{i+1}", priority=2)

print("Order jobs are processed (watch HIGH priority jump ahead of LOW):")
time.sleep(2.5)
pool3.stop()

print()
stats3 = pool3.stats()
if stats3:
    print(f"Total completed: {stats3['completed']}")
    print(f"Queue rejected:  {stats3['queue_rejected']}")
    print()
    print("By priority group:")
    for label, s in stats3.get("by_priority", {}).items():
        print(f"  {label}: {s['count']} jobs, avg {s['avg_latency_ms']:.0f}ms/job")

# ── Queue depth as scaling signal ─────────────────────────────────────────────
print()
print("=== Queue Depth as Autoscaling Signal ===")
print("Rule: queue_depth > threshold -> scale out; depth=0 -> scale in")
print()
depth_thresholds = [(0, "scale_in"), (3, "hold"), (8, "scale_out"), (15, "scale_out_urgent")]
print(f"{'Queue Depth':>12} {'Action':>18} {'Why'}")
print("-" * 50)
for thresh, action in depth_thresholds:
    why = ("Workers keeping up" if thresh == 0 else
           "Normal operation" if thresh == 3 else
           "Workers falling behind" if thresh == 8 else
           "Serious lag - add workers NOW")
    print(f"{thresh:>12}   {action:>18}   {why}")

## Demo 4: Request Batching — Getting More from the Same Hardware

Instead of one request at a time, **batch multiple requests** and process together.

| Strategy | How | Downside |
|----------|-----|---------|
| **Static batching** | Wait for N requests, then process all | First request waits for N-1 more to arrive |
| **Dynamic batching** | Wait up to T milliseconds, process whatever arrived | Long responses block short ones |
| **Continuous batching** | Return completed responses immediately, swap in waiting requests | More complex, highest throughput |

> **In Simple Words:**
> Static batching = wait for the bus to fill before departing (everyone waits).
> Dynamic batching = bus departs every 5 minutes regardless of how many aboard.
> Continuous batching = bus stops at each shelter, lets passengers on/off simultaneously.
> Continuous batching is how vLLM/TGI achieve 3-5x higher throughput than naive serving.

In [ ]:
# ── Demo 4: Request Batching Strategies ──────────────────────────────────────

def process_single(prompt: str) -> tuple:
    'Baseline: process one request at a time.'
    text, lat_ms, tok_in, tok_out = llm_call(prompt)
    return text, lat_ms

def process_static_batch(prompts: list, batch_size: int = 4) -> list:
    '''
    Static batching: wait for batch_size requests then process all.
    Returns list of (text, latency_ms) per prompt.
    First request waits until batch is full.
    '''
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        batch_results = []
        # Simulate parallel processing within a batch (threading)
        threads = []
        lock = threading.Lock()

        def do_call(p):
            text, lat_ms, _, _ = llm_call(p)
            with lock:
                batch_results.append((text, lat_ms))

        for p in batch:
            t = threading.Thread(target=do_call, args=(p,))
            threads.append(t)
        for t in threads: t.start()
        for t in threads: t.join()
        results.extend(batch_results)
    return results


def process_dynamic_batch(prompts: list, wait_ms: float = 50) -> list:
    '''
    Dynamic batching: collect requests for wait_ms then process together.
    Batches are whatever arrived in the window.
    '''
    results = []
    # Simulate requests arriving over time (10ms apart)
    arrival_queue = deque(prompts)
    current_batch = []
    batch_deadline = time.time() + wait_ms / 1000

    while arrival_queue or current_batch:
        # Accept arrivals for this window
        while arrival_queue and time.time() < batch_deadline:
            current_batch.append(arrival_queue.popleft())
            time.sleep(0.001)   # 1ms between arrivals (scaled)

        if current_batch:
            # Process the batch in parallel
            batch_results = []
            lock = threading.Lock()

            def do_call(p):
                text, lat_ms, _, _ = llm_call(p)
                with lock:
                    batch_results.append((text, lat_ms))

            threads = [threading.Thread(target=do_call, args=(p,))
                       for p in current_batch]
            for t in threads: t.start()
            for t in threads: t.join()
            results.extend(batch_results)
            current_batch = []
        batch_deadline = time.time() + wait_ms / 1000
    return results


def process_continuous_batch(prompts: list, batch_slots: int = 3) -> list:
    '''
    Continuous batching simulation:
    - Maintain batch_slots concurrent slots
    - When a slot completes, immediately fill it with the next waiting request
    - Short requests complete and free their slot without waiting for long ones
    '''
    results = []
    lock = threading.Lock()
    pending = deque(prompts)
    active_count = [0]
    done_event = threading.Event()

    def process_one(prompt, slot_id):
        text, lat_ms, _, _ = llm_call(prompt)
        with lock:
            results.append((text, lat_ms))
            active_count[0] -= 1
            # Immediately start next job in this slot
            if pending:
                next_prompt = pending.popleft()
                active_count[0] += 1
                t = threading.Thread(target=process_one, args=(next_prompt, slot_id))
                t.daemon = True
                t.start()
            elif active_count[0] == 0:
                done_event.set()

    # Seed the batch slots
    with lock:
        for _ in range(min(batch_slots, len(pending))):
            p = pending.popleft()
            active_count[0] += 1
            t = threading.Thread(target=process_one, args=(p, _))
            t.daemon = True
            t.start()

    done_event.wait(timeout=30)
    return results


# ── Compare strategies on 12 requests ─────────────────────────────────────────
random.seed(7)
test_prompts = (
    [f"What is BGP {attr}?" for attr in
     ["LOCAL_PREF", "AS_PATH", "MED", "ORIGIN", "NEXT_HOP", "COMMUNITY"]] +
    [f"Explain OSPF {topic}" for topic in
     ["DR election", "area types", "hello packets", "LSA flooding", "stub areas", "NSSA"]]
)

print("=== Batching Strategy Comparison (12 requests) ===")
print()

# Strategy 1: Sequential (baseline)
start = time.time()
seq_results = [process_single(p) for p in test_prompts]
seq_time = (time.time() - start) * 1000
seq_lats = [r[1] for r in seq_results]
print(f"Sequential (1 at a time):     {seq_time:>7.0f}ms total, "
      f"avg {sum(seq_lats)/len(seq_lats):.0f}ms/req")

# Strategy 2: Static batching (batch_size=4)
start = time.time()
static_results = process_static_batch(test_prompts, batch_size=4)
static_time = (time.time() - start) * 1000
static_lats = [r[1] for r in static_results]
speedup_static = seq_time / static_time if static_time else 1
print(f"Static batch (size=4):        {static_time:>7.0f}ms total, "
      f"speedup={speedup_static:.1f}x")

# Strategy 3: Dynamic batching (50ms window)
start = time.time()
dynamic_results = process_dynamic_batch(test_prompts, wait_ms=30)
dynamic_time = (time.time() - start) * 1000
speedup_dynamic = seq_time / dynamic_time if dynamic_time else 1
print(f"Dynamic batch (30ms window):  {dynamic_time:>7.0f}ms total, "
      f"speedup={speedup_dynamic:.1f}x")

# Strategy 4: Continuous batching (3 slots)
start = time.time()
cont_results = process_continuous_batch(test_prompts, batch_slots=3)
cont_time = (time.time() - start) * 1000
speedup_cont = seq_time / cont_time if cont_time else 1
print(f"Continuous batch (3 slots):   {cont_time:>7.0f}ms total, "
      f"speedup={speedup_cont:.1f}x  <- best!")

print()
print("Why continuous batching wins:")
print("  - Short requests complete and free slots immediately")
print("  - Slots never sit idle waiting for slow requests to finish")
print("  - In vLLM: batch_slots = GPU's concurrent sequence capacity")
print("  - 3-5x throughput improvement vs static batching on mixed workloads")

## Demo 5: Reactive Autoscaler — Scale on Queue Depth

**Queue depth is the best autoscaling signal for AI workloads.**

Why NOT CPU utilization? An async AI service spends most time waiting for LLM API
responses, not computing. CPU can be low while the service is completely overwhelmed.

**Queue depth tells you directly:** "N jobs are waiting. Add more workers."

**Autoscaler rules:**
- Queue depth > `scale_out_threshold` AND growing -> add workers
- Queue depth = 0 for `scale_in_delay` seconds -> remove a worker (save cost)
- `min_workers` floor, `max_workers` ceiling

**Metastable failure prevention:**
Queue flooding + retry storm = system never recovers even after adding capacity.
Fix: backpressure (429) + exponential backoff + autoscale together.

> **In Simple Words:** Autoscaler = OSPF route dampening + automatic failover.
> Queue depth growing = dampening threshold exceeded -> add routes (workers).
> Queue depth zero for 5 minutes = stable, safe to remove backup routes.

In [ ]:
# ── Demo 5: Reactive Autoscaler ──────────────────────────────────────────────

class ReactiveAutoscaler:
    '''
    Monitors queue depth and dynamically adjusts number of active workers.

    Scale-out trigger: queue_depth > scale_out_threshold for scale_window checks
    Scale-in trigger:  queue_depth == 0 for scale_in_delay seconds
    Cooldown:          wait cooldown_s after each scale event before next
    '''
    def __init__(self, min_workers: int = 1, max_workers: int = 8,
                 scale_out_threshold: int = 5, scale_in_delay: float = 3.0,
                 cooldown_s: float = 1.0):
        self.min_workers         = min_workers
        self.max_workers         = max_workers
        self.scale_out_threshold = scale_out_threshold
        self.scale_in_delay      = scale_in_delay
        self.cooldown_s          = cooldown_s

        self._current_workers = min_workers
        self._last_scale_time = 0.0
        self._idle_since      = None   # time when queue went to 0
        self._scale_events    = []     # log of scale events
        self._snapshots       = []     # (time, depth, workers)

    def _can_scale(self) -> bool:
        return (time.time() - self._last_scale_time) >= self.cooldown_s

    def observe(self, queue_depth: int) -> str:
        'Feed current queue depth. Returns action taken: "scale_out", "scale_in", or "hold".'
        now = time.time()
        self._snapshots.append((now, queue_depth, self._current_workers))
        action = "hold"

        if not self._can_scale():
            return "cooldown"

        if queue_depth > self.scale_out_threshold and                 self._current_workers < self.max_workers:
            # Scale out
            self._current_workers += 1
            self._last_scale_time = now
            self._idle_since = None
            event = f"SCALE OUT -> {self._current_workers} workers (q={queue_depth})"
            self._scale_events.append(event)
            action = "scale_out"

        elif queue_depth == 0 and self._current_workers > self.min_workers:
            if self._idle_since is None:
                self._idle_since = now
            elif (now - self._idle_since) >= self.scale_in_delay:
                # Scale in after sustained idle period
                self._current_workers -= 1
                self._last_scale_time = now
                self._idle_since = None
                event = f"SCALE IN  -> {self._current_workers} workers (idle {self.scale_in_delay}s)"
                self._scale_events.append(event)
                action = "scale_in"
        else:
            self._idle_since = None

        return action

    @property
    def workers(self) -> int:
        return self._current_workers

    def summary(self) -> dict:
        depths  = [s[1] for s in self._snapshots]
        workers = [s[2] for s in self._snapshots]
        return {
            "scale_events":    self._scale_events,
            "max_workers_seen": max(workers) if workers else 0,
            "min_workers_seen": min(workers) if workers else 0,
            "avg_queue_depth": round(sum(depths)/len(depths), 1) if depths else 0,
            "max_queue_depth": max(depths) if depths else 0,
        }


# ── Simulate a realistic traffic pattern ─────────────────────────────────────
# Traffic profile: ramp up -> peak -> sustained -> drain -> scale in
def generate_traffic_profile():
    'Returns list of (time_s, arrivals_per_second) tuples.'
    profile = []
    # t=0-5: ramp up
    for t in range(6):
        profile.append((t, t * 1.5))
    # t=5-15: peak (batch job floods in)
    for t in range(5, 16):
        profile.append((t, 10 + random.uniform(-1, 2)))
    # t=15-20: taper
    for t in range(15, 21):
        profile.append((t, max(0, 10 - (t-15) * 2)))
    # t=20-25: quiet
    for t in range(20, 26):
        profile.append((t, 0.5))
    return profile

random.seed(11)
scaler = ReactiveAutoscaler(
    min_workers=1, max_workers=6,
    scale_out_threshold=4, scale_in_delay=2.0, cooldown_s=0.5
)

print("=== Reactive Autoscaler Simulation ===")
print(f"Config: min={scaler.min_workers}, max={scaler.max_workers}, "
      f"scale_out if depth>{scaler.scale_out_threshold}, "
      f"scale_in after {scaler.scale_in_delay}s idle")
print()

# Simulate time steps with a queue that responds to worker count
traffic_profile = generate_traffic_profile()
simulated_queue_depth = 0
processing_rate_per_worker = 2.0   # jobs/second per worker

print(f"{'Time':>5} {'Arrivals/s':>10} {'Workers':>8} {'Q Depth':>8} {'Action':>12}")
print("-" * 50)

for t, arrivals_s in traffic_profile:
    # Queue fills based on arrival rate vs processing capacity
    capacity = scaler.workers * processing_rate_per_worker
    net_change = arrivals_s - capacity
    simulated_queue_depth = max(0, simulated_queue_depth + net_change * 0.5)
    simulated_queue_depth = round(simulated_queue_depth)

    # Autoscaler observes and acts
    action = scaler.observe(int(simulated_queue_depth))

    action_str = ""
    if action == "scale_out":
        action_str = "SCALE OUT +"
    elif action == "scale_in":
        action_str = "SCALE IN  -"
    elif action == "cooldown":
        action_str = "(cooldown)"

    print(f"t={t:>3}s  {arrivals_s:>9.1f}   {scaler.workers:>7}   "
          f"{simulated_queue_depth:>7}   {action_str}")

print()
summary = scaler.summary()
print("=== Autoscaler Summary ===")
print(f"  Max workers deployed: {summary['max_workers_seen']}")
print(f"  Final workers:        {scaler.workers}")
print(f"  Max queue depth:      {summary['max_queue_depth']}")
print(f"  Avg queue depth:      {summary['avg_queue_depth']}")
print(f"  Scale events ({len(summary['scale_events'])}):")
for event in summary['scale_events']:
    print(f"    {event}")

# ── Metastable failure warning ─────────────────────────────────────────────────
print()
print("=== Metastable Failure Prevention ===")
print("Scenario: 200 clients timeout -> immediately retry -> system can't recover")
print()
print("WITHOUT protection:")
print("  1. System overloaded, clients timeout (30s)")
print("  2. Clients immediately retry -> 2x original load")
print("  3. New workers come online but retry storm overwhelms them too")
print("  4. System stays saturated indefinitely")
print()
print("WITH three-layer protection:")
protections = [
    ("Backpressure (429)",          "Reject new requests when queue full - clients STOP sending"),
    ("Exponential backoff",         "Each retry waits 2x longer - load naturally drops off"),
    ("Autoscale on queue depth",    "Add workers proportional to real demand, not retry storms"),
]
for layer, effect in protections:
    print(f"  Layer: {layer}")
    print(f"    -> {effect}")
    print()

## Summary — What You Built

| Demo | Pattern | Key Insight |
|------|---------|-------------|
| 1 | Worker Pool + DLQ | Decouple arrival from processing; failed jobs quarantined, not looping |
| 2 | Load Balancer | Least-connections beats round-robin for mixed fast/slow LLM requests |
| 3 | Priority Queue | Real-time queries skip batch jobs; queue depth drives scaling decisions |
| 4 | Request Batching | Continuous batching: 3x speedup vs sequential, no idle GPU slots |
| 5 | Reactive Autoscaler | Scale on queue depth (not CPU); three-layer metastable failure prevention |

### LLM Bottleneck Cheat Sheet
```
Prefill phase  (compute-bound):   long prompts/context -> add GPU FLOP/s
Decode phase   (memory-bound):    output generation -> increase memory bandwidth
Throughput:                        more requests -> horizontal scale + batching
Single-req latency:                Amdahl's Law ceiling -> speculative decoding
```

### Scaling Decision Tree
```
Queue depth growing?   YES -> scale out (add workers)
Queue depth = 0?       YES, sustained -> scale in (save cost)
p95 latency spiking?   YES -> check if batch/real-time mixed (priority queue)
Cost spike?            YES -> check for retry storm or agent loop (backpressure)
```

### Production Tech Stack (what these demos simulate)
| Demo Component | Production Equivalent |
|---------------|----------------------|
| `queue_module.Queue` | Redis Streams, Kafka, RabbitMQ |
| `threading.Thread` workers | Kubernetes pods, ECS tasks |
| `LoadBalancer` class | nginx, HAProxy, AWS ALB |
| `ReactiveAutoscaler` | Kubernetes HPA, KEDA |
| Static/continuous batching | vLLM, Hugging Face TGI |

In [ ]:
# ── Integration: Full Scaling Stack for Network AI API ───────────────────────

print("=" * 65)
print("INTEGRATION: Complete Scaling Stack for Network AI Operations")
print("=" * 65)

random.seed(55)

# Build the full stack
main_pool    = WorkerPool(num_workers=3, queue_max=20)
lb           = LoadBalancer(num_instances=3, strategy="least_connections")
autoscaler   = ReactiveAutoscaler(min_workers=2, max_workers=5,
                                   scale_out_threshold=5, scale_in_delay=1.5)

main_pool.start()

# Traffic mix: batch audit (LOW) + NOC queries (HIGH) + monitoring (MED)
traffic = [
    # (prompt, priority, delay_before_ms)
    ("Batch: check OSPF config device-01",   1,  0),
    ("Batch: check OSPF config device-02",   1, 10),
    ("URGENT: BGP down on core-rtr-01",      3, 20),
    ("Batch: check OSPF config device-03",   1, 25),
    ("URGENT: BGP down on core-rtr-02",      3, 30),
    ("Monitor: QoS status dist-sw-01",       2, 35),
    ("Batch: check OSPF config device-04",   1, 40),
    ("Monitor: QoS status dist-sw-02",       2, 45),
    ("Batch: check OSPF config device-05",   1, 50),
    ("URGENT: Interface down on access-sw",  3, 55),
    ("Batch: check OSPF config device-06",   1, 60),
    ("Batch: check OSPF config device-07",   1, 65),
    ("Monitor: BGP summary all routers",     2, 70),
    ("URGENT: Route leak detected",          3, 75),
    ("Batch: check OSPF config device-08",   1, 80),
]

print()
print(f"{'#':>3} {'Priority':>9} {'Submit':>8} {'Result':>14}")
print("-" * 40)

priority_labels = {3: "HIGH", 2: "MED ", 1: "LOW "}
for i, (prompt, priority, delay_ms) in enumerate(traffic, 1):
    time.sleep(delay_ms / 1000)
    job_id, status = main_pool.submit(prompt, priority=priority)
    label = priority_labels.get(priority, "?")
    print(f"{i:>3} [{label}]  {delay_ms:>5}ms   {status}")
    # Observe queue depth for autoscaler
    depth = main_pool.queue_depth()
    autoscaler.observe(depth)

print()
print("Processing... (1.5 seconds)")
time.sleep(1.5)
main_pool.stop()
time.sleep(0.3)

# Final report
print()
print("=== Final System Report ===")
pool_report = main_pool.report()
print(f"Completed jobs:    {pool_report['completed']}")
print(f"DLQ failures:      {pool_report['dlq_failures']}")
print(f"Avg LLM latency:   {pool_report['avg_latency_ms']:.0f} ms")
print(f"Avg queue wait:    {pool_report['avg_wait_ms']:.0f} ms")

scale_summary = autoscaler.summary()
print(f"Peak workers used: {scale_summary['max_workers_seen']}")
print(f"Scale events:      {len(scale_summary['scale_events'])}")

print()
print("In production this stack provides:")
print("  - Bounded queue prevents OOM crashes under load spikes")
print("  - Priority queue: real-time queries never wait behind batch jobs")
print("  - LB least-connections: fast requests don't queue behind slow ones")
print("  - Autoscaler adds workers during batch spikes, saves cost at night")
print("  - DLQ isolates bad jobs: one malformed request can't block the whole queue")